# Exemplo 1 — Self-RAG Mínimo com vLLM

**Aula 8 · MBA em RAG & CAG Aplicados a Direito e Segurança Pública**

---

## Objetivo

Demonstrar o conceito central do Self-RAG: o modelo decide *quando* recuperar documentos usando tokens de controle. Vamos simular esse comportamento de forma mínima com um LLM local via **vLLM** (servidor OpenAI-compatible), emulando os tokens `[Retrieve]`, `[ISREL]`, `[ISSUP]` e `[ISUSE]`.

## O que você vai aprender

- Como os tokens de controle do Self-RAG funcionam na prática
- Como subir um servidor vLLM no Colab e consumi-lo via API OpenAI
- Como simular recuperação seletiva (apenas quando necessário)
- Diferença entre queries que ativam e que não ativam o retrieval

> **Nota:** Este exemplo usa uma implementação simplificada que *emula* o Self-RAG real. O modelo Self-RAG genuíno (`llama-2-7b-selfrag`) via vLLM é apresentado no LAB1.

## Passo 1 — Instalação das Dependências

O **vLLM** é um servidor de inferência de alta performance para LLMs que expõe uma API compatível com OpenAI. Isso permite usar o `langchain-openai` para consumir modelos locais sem alterações no código.

In [ ]:
# Instalar dependências necessárias
# vLLM: servidor de inferência OpenAI-compatible para LLMs open-source
# langchain-openai: cliente que consome qualquer servidor OpenAI-compatible
!pip install vllm langchain langchain-community langchain-openai chromadb sentence-transformers -q

print("Instalação concluída!")

In [ ]:
import subprocess
import time
import os

# ─── Subir o servidor vLLM em background ──────────────────────────────────────
# O vLLM baixa o modelo do HuggingFace automaticamente na primeira execução.
# Usamos Meta-Llama-3.2-1B-Instruct (1B parâmetros) para rodar no Colab T4.
# Para o modelo Self-RAG genuíno: substitua por 'selfrag/selfrag_llama2_7b'

MODELO = "meta-llama/Llama-3.2-1B-Instruct"  # Leve, ideal para demonstração
PORTA  = 8000

print(f"Iniciando servidor vLLM com o modelo '{MODELO}'...")
print("Aguarde o download e carregamento (pode levar alguns minutos na primeira vez)")

# Inicia o servidor vLLM como subprocesso em background
# --dtype auto: detecta automaticamente float16 ou bfloat16 conforme a GPU
# --max-model-len 4096: limita o contexto para economizar VRAM no Colab
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODELO,
        "--port", str(PORTA),
        "--dtype", "auto",
        "--max-model-len", "4096",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Aguardar o servidor ficar disponível (verifica a cada 5s, timeout de 3min)
import urllib.request

for tentativa in range(36):  # 36 * 5s = 3 minutos
    try:
        urllib.request.urlopen(f"http://localhost:{PORTA}/health")
        print(f"\nServidor vLLM pronto na porta {PORTA}!")
        break
    except Exception:
        print(f"  Aguardando servidor... ({(tentativa+1)*5}s)", end="\r")
        time.sleep(5)
else:
    print("\nATENÇÃO: Servidor pode ainda não estar pronto. Continue mesmo assim.")

## Passo 2 — Construir o Corpus Jurídico

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

# Corpus jurídico simplificado
documentos_juridicos = [
    Document(
        page_content="Art. 206, §3º, V do Código Civil: prescreve em 3 anos a pretensão de reparação civil.",
        metadata={"fonte": "Código Civil", "artigo": "206"}
    ),
    Document(
        page_content="Art. 186 do Código Civil: aquele que causar dano a outrem, ainda que moral, comete ato ilícito.",
        metadata={"fonte": "Código Civil", "artigo": "186"}
    ),
    Document(
        page_content="Súmula 227 STJ: A pessoa jurídica pode sofrer dano moral.",
        metadata={"fonte": "STJ", "tipo": "sumula"}
    ),
    Document(
        page_content="Art. 302 CPP: configura flagrante delito quem está cometendo a infração, acabou de cometê-la, é perseguido logo após, ou encontrado logo depois com instrumentos do crime.",
        metadata={"fonte": "CPP", "artigo": "302"}
    ),
]

# Criar índice vetorial com embeddings multilíngues
print("Criando índice vetorial...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vectorstore = Chroma.from_documents(documentos_juridicos, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Índice criado com {len(documentos_juridicos)} documentos.")

## Passo 3 — Conectar ao Servidor vLLM e Implementar Tokens de Controle

O vLLM expõe uma API 100% compatível com OpenAI. Basta apontar `base_url` para `localhost` e usar `langchain-openai` normalmente — o modelo local se comporta como o GPT.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage
import json

# ─── Inicializar o cliente apontando para o servidor vLLM local ───────────────
# openai_api_key="EMPTY": o vLLM local não exige chave real
# openai_api_base: URL do servidor vLLM
# model_name: deve corresponder ao modelo carregado no servidor
llm = ChatOpenAI(
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    openai_api_key="EMPTY",
    openai_api_base=f"http://localhost:{PORTA}/v1",
    temperature=0,
    max_tokens=256,
)

# Teste rápido de conectividade
print("Testando conexão com o servidor vLLM...")
teste = llm.invoke("Responda em uma palavra: qual é a capital do Brasil?")
print(f"Resposta de teste: {teste.content.strip()}")
print("Conexão OK!")

In [ ]:
# ─── Funções dos Tokens de Controle ──────────────────────────────────────────
# Cada função emula um token Self-RAG chamando o LLM com um prompt específico.

def token_retrieve(pergunta: str) -> bool:
    """
    Simula o token [Retrieve]: decide se a pergunta requer busca em documentos.
    Retorna True se deve recuperar, False se o modelo responde com conhecimento próprio.
    """
    prompt = (
        "Você é um classificador. Dado uma pergunta, decida se ela requer busca "
        "em uma base de documentos jurídicos específicos.\n\n"
        "Responda APENAS com 'SIM' se a pergunta:\n"
        "- Pede artigos, prazos, penalidades ou textos legais específicos\n"
        "- Pede jurisprudência ou súmulas específicas\n"
        "- Requer fatos específicos de documentos\n\n"
        "Responda APENAS com 'NAO' se a pergunta:\n"
        "- Pede definições gerais de conceitos jurídicos\n"
        "- Pede explicação de princípios gerais do direito\n"
        "- Pode ser respondida com conhecimento geral\n\n"
        f"Pergunta: {pergunta}\n\nResposta (SIM ou NAO):"
    )
    resposta = llm.invoke(prompt).content.strip().upper()
    return "SIM" in resposta


def token_isrel(pergunta: str, documento: str) -> str:
    """
    Simula o token [ISREL]: avalia se o documento é relevante para a pergunta.
    Retorna 'relevant' ou 'irrelevant'.
    """
    prompt = (
        f"Avalie a relevância do documento abaixo para responder à pergunta.\n\n"
        f"Pergunta: {pergunta}\nDocumento: {documento}\n\n"
        "O documento é relevante? Responda apenas 'relevant' ou 'irrelevant'."
    )
    resposta = llm.invoke(prompt).content.strip().lower()
    return "relevant" if ("relevant" in resposta and "irr" not in resposta) else "irrelevant"


def token_issup(documento: str, geracao: str) -> str:
    """
    Simula o token [ISSUP]: verifica se a geração tem suporte no documento.
    Retorna 'fully supported', 'partially supported' ou 'no support'.
    """
    prompt = (
        f"Verifique se a resposta gerada tem suporte factual no documento.\n\n"
        f"Documento: {documento}\nResposta gerada: {geracao}\n\n"
        "Responda apenas: 'fully supported', 'partially supported' ou 'no support'."
    )
    resposta = llm.invoke(prompt).content.strip().lower()
    if "fully" in resposta:
        return "fully supported"
    elif "partial" in resposta:
        return "partially supported"
    return "no support"


def token_isuse(pergunta: str, geracao: str) -> int:
    """
    Simula o token [ISUSE]: avalia a utilidade da resposta (escala 1-5).
    """
    prompt = (
        f"Avalie a utilidade da resposta para a pergunta em uma escala de 1 a 5.\n\n"
        f"Pergunta: {pergunta}\nResposta: {geracao}\n\n"
        "5=excelente, 4=boa, 3=razoável, 2=fraca, 1=inútil\n"
        "Responda APENAS com o número (1, 2, 3, 4 ou 5):"
    )
    resposta = llm.invoke(prompt).content.strip()
    for char in resposta:
        if char in '12345':
            return int(char)
    return 3  # valor padrão


print("Funções de tokens de controle definidas com sucesso!")

## Passo 4 — Pipeline Self-RAG Completo

In [ ]:
def self_rag_pipeline(pergunta: str) -> dict:
    """
    Pipeline Self-RAG simplificado com todos os tokens de controle.
    
    Fluxo:
      1. [Retrieve]: decide se busca documentos
      2. [ISREL]:    filtra documentos relevantes
      3. Gera resposta com (ou sem) contexto
      4. [ISSUP]:    verifica suporte factual
      5. [ISUSE]:    avalia utilidade da resposta
    """
    tokens_emitidos = {}
    documentos_usados = []

    print(f"\n{'='*60}")
    print(f"PERGUNTA: {pergunta}")
    print('='*60)

    # ── Token 1: [Retrieve] ───────────────────────────────────────
    deve_recuperar = token_retrieve(pergunta)
    tokens_emitidos['[Retrieve]'] = 'yes' if deve_recuperar else 'no'
    print(f"[Retrieve] = {tokens_emitidos['[Retrieve]']}")

    contexto = ""

    if deve_recuperar:
        # ── Recuperação ────────────────────────────────────────────
        docs = retriever.invoke(pergunta)
        print(f"  → {len(docs)} documentos recuperados")

        # ── Token 2: [ISREL] para cada documento ───────────────────
        docs_relevantes = []
        for i, doc in enumerate(docs):
            relevancia = token_isrel(pergunta, doc.page_content)
            tokens_emitidos[f'[ISREL_doc{i+1}]'] = relevancia
            print(f"  [ISREL doc{i+1}] = {relevancia}: {doc.page_content[:60]}...")
            if relevancia == "relevant":
                docs_relevantes.append(doc)
                documentos_usados.append(doc.page_content)

        if docs_relevantes:
            contexto = "\n".join([d.page_content for d in docs_relevantes])
        else:
            print("  ⚠ Nenhum documento relevante — gerando sem contexto")

    # ── Geração ────────────────────────────────────────────────────
    if contexto:
        prompt_geracao = (
            f"Baseando-se nos documentos jurídicos abaixo, responda à pergunta "
            f"de forma precisa e objetiva.\n\nDocumentos:\n{contexto}\n\n"
            f"Pergunta: {pergunta}\n\nResposta:"
        )
    else:
        prompt_geracao = (
            f"Responda à pergunta abaixo usando seu conhecimento geral sobre "
            f"direito brasileiro.\n\nPergunta: {pergunta}\n\nResposta:"
        )

    geracao = llm.invoke(prompt_geracao).content.strip()

    # ── Token 3: [ISSUP] ───────────────────────────────────────────
    if contexto and documentos_usados:
        suporte = token_issup(contexto, geracao)
        tokens_emitidos['[ISSUP]'] = suporte
        print(f"[ISSUP] = {suporte}")

    # ── Token 4: [ISUSE] ───────────────────────────────────────────
    utilidade = token_isuse(pergunta, geracao)
    tokens_emitidos['[ISUSE]'] = utilidade
    print(f"[ISUSE] = {utilidade}/5")

    print(f"\nRESPOSTA FINAL:")
    print(geracao)

    return {
        "resposta": geracao,
        "tokens": tokens_emitidos,
        "documentos_usados": documentos_usados
    }


print("Pipeline Self-RAG pronto!")

## Passo 5 — Testando com Queries que ATIVAM o Retrieval

In [ ]:
# Query 1: Requer documento específico → [Retrieve=yes]
resultado1 = self_rag_pipeline(
    "Qual o prazo prescricional para ação de indenização por danos morais no Brasil?"
)

In [ ]:
# Query 2: Requer artigo específico → [Retrieve=yes]
resultado2 = self_rag_pipeline(
    "O que configura flagrante delito segundo o Código de Processo Penal?"
)

## Passo 6 — Testando com Queries que NÃO ATIVAM o Retrieval

In [ ]:
# Query 3: Conhecimento geral → [Retrieve=no]
resultado3 = self_rag_pipeline(
    "O que é responsabilidade civil?"
)

In [ ]:
# Query 4: Conceito geral → [Retrieve=no]
resultado4 = self_rag_pipeline(
    "Qual a diferença entre crime doloso e culposo?"
)

## Passo 7 — Comparativo de Tokens Emitidos

In [ ]:
import pandas as pd

resumo = [
    {"Query": "Prazo prescricional danos morais",  "[Retrieve]": resultado1['tokens'].get('[Retrieve]', '-'), "[ISSUP]": resultado1['tokens'].get('[ISSUP]', 'N/A'), "[ISUSE]": resultado1['tokens'].get('[ISUSE]', '-')},
    {"Query": "Flagrante delito CPP",              "[Retrieve]": resultado2['tokens'].get('[Retrieve]', '-'), "[ISSUP]": resultado2['tokens'].get('[ISSUP]', 'N/A'), "[ISUSE]": resultado2['tokens'].get('[ISUSE]', '-')},
    {"Query": "O que é responsabilidade civil",    "[Retrieve]": resultado3['tokens'].get('[Retrieve]', '-'), "[ISSUP]": resultado3['tokens'].get('[ISSUP]', 'N/A'), "[ISUSE]": resultado3['tokens'].get('[ISUSE]', '-')},
    {"Query": "Crime doloso vs culposo",           "[Retrieve]": resultado4['tokens'].get('[Retrieve]', '-'), "[ISSUP]": resultado4['tokens'].get('[ISSUP]', 'N/A'), "[ISUSE]": resultado4['tokens'].get('[ISUSE]', '-')},
]

df = pd.DataFrame(resumo)
print("\n=== COMPARATIVO DE TOKENS SELF-RAG ===")
print(df.to_string(index=False))
print("\nObservação: Queries específicas ativam [Retrieve=yes].")
print("            Queries de conhecimento geral ativam [Retrieve=no].")

## Conclusão

Neste exemplo você viu como o Self-RAG usa tokens de controle para:

1. **`[Retrieve]`** — decidir inteligentemente quando buscar documentos
2. **`[ISREL]`** — filtrar documentos recuperados por relevância
3. **`[ISSUP]`** — verificar se a resposta tem suporte factual
4. **`[ISUSE]`** — avaliar a utilidade da resposta gerada

O servidor **vLLM** serviu como backend de inferência, expondo uma API OpenAI-compatible que permitiu integrar qualquer LLM open-source sem mudar o código do pipeline.

No **LAB1**, você usará o modelo `selfrag/selfrag_llama2_7b` genuíno via vLLM, onde os tokens são emitidos nativamente pelo próprio modelo durante a geração — sem chamadas LLM separadas para cada avaliação.

---

*MBA em RAG & CAG Aplicados a Direito e Segurança Pública · Aula 8*